In [ ]:
library(Seurat)
Sys.setenv(HDF5_DISABLE_VERSION_CHECK="2")
library(SeuratDisk)

In [ ]:
combined_loom_dir <- "/PATH/combined_loom/"
combined_process_loom_dir <- "/PATH/combined_process_loom/"
compass_out_dir <- "/PATH/compass_out/"

#### SAMPLE.combined

#### 1. preprocess and plot pie chart of each clone

In [ ]:
## 1. get cell metadata
sample <- "SAMPLE.combined"

loom_obj <- Connect(filename = paste0(combined_loom_dir, sample, ".loom"), mode = "r")

cell_attr_names <- names(loom_obj[["col_attrs"]])
print(cell_attr_names)

cell_metadata <- data.frame(
  lapply(cell_attr_names, function(x) loom_obj[["col_attrs"]][[x]][])
)

colnames(cell_metadata) <- cell_attr_names
cell_metadata$Stage <- sub("_.*", "", cell_metadata$barcode)

head(cell_metadata)
outdir <- paste0(compass_out_dir, sample, "/plot/")
dir.create(outdir, showWarnings = FALSE, recursive = TRUE)
write.csv(cell_metadata, file = paste0(outdir, sample, "_cell_metadata.csv"), row.names = FALSE)

loom_obj$close_all()

In [ ]:
##  2. match compass out cell stage info
outdir <- paste0(compass_out_dir, sample, "/plot/")
df_stage <- read.csv(paste0(outdir, sample, "_cell_metadata.csv"))
df_index <- read.table(paste0(combined_process_loom_dir, sample, "/SAMPLE.combined_cell_barcodes.txt"),sep="\t")
df_index$bc_index <- seq(0,nrow(df_index)-1)

df_match <- merge(df_stage,df_index,by.x="barcode",by.y="V1",all.y=TRUE)

df_node <- read.csv(paste0(compass_out_dir, sample, "/", sample, "_cellAssignments.tsv"),sep="\t")
df_node_info <- merge(df_node,df_match,by.x="cell",by.y="bc_index",all.x=TRUE)
df_node_info_qc <- df_node_info[which(df_node_info$doublet == "no"),]
write.csv(df_node_info_qc, paste0(outdir, sample, "_cell_stage.csv"))

df_node_variant <- read.csv(paste0(compass_out_dir, sample, "/", sample, "_nodes_genotypes.tsv"),sep="\t")
# match node name with variants
df_node_variant$node_num <- sapply(df_node_variant$node, function(x) {
  strsplit(x, " ")[[1]][2]
})
df_node_variant <- as.data.frame(df_node_variant)
write.csv(df_node_variant, paste0(outdir, sample, "_nodes_genotypes.csv"), row.names = FALSE)

# plot pie chart
df_pie <- df_node_info_qc %>%
  group_by(node,Stage) %>%
  summarise(count = n()) %>%
  mutate(prop = count / sum(count))
df_pie$node <- as.character(df_pie$node)
df_pie <- df_pie |>
  dplyr::left_join(
    df_node_variant |> dplyr::select(node_num, match_name),
    by = c("node" = "node_num")
  )
df_pie$node_genotype <- paste0(df_pie$node, " (", df_pie$match_name, ")")
df_pie$Stage <- factor(df_pie$Stage, levels = c("MRD", "IDX"))

library(ggplot2)
library(scales)
p <- ggplot(df_pie, aes(x = "", y = prop, fill = Stage)) +
  geom_bar(stat = "identity", width = 1, color = "white") + 
  coord_polar(theta = "y", start = 0) +
  facet_wrap(~ paste0("Node ", node), ncol = 3) + 
  geom_text(aes(label = percent(prop, accuracy = 0.1)), 
            position = position_stack(vjust = 0.5), 
            size = 8) +
  theme_void() + 
  labs(title = "Proportion of Stage (IDX vs MRD) across Clonal Nodes",
       fill = "Stage") +
  theme(strip.text = element_text(size = 12, face = "bold", margin = margin(b = 5)), legend.position = "right")
ggsave(p, filename = paste0(outdir, sample, "_node_stage_pie.pdf"), height = 8, width = 12)

In [ ]:
df_test <- df_node_info_qc %>%
  group_by(Stage,node) %>%
  summarise(count = n()) %>%
  mutate(prop = count / sum(count))

#### 2. plot fishplot

In [ ]:
df_count <- df_node_info_qc %>%
  group_by(node,Stage) %>%
  summarise(count = n())
df_count$node <- as.character(df_count$node)

In [ ]:
library(dplyr)
library(timescape)

# ==========================================
# 1. calculate clonal prevalence for each stage
# ==========================================
clonal_prev_base <- df_count %>%
  group_by(Stage) %>%
  mutate(
    total_cells = sum(count),
    obs_prop = count / total_cells
  ) %>%
  ungroup() %>%
  select(timepoint = Stage, clone_id = node, obs_prop)

clonal_prev_base$timepoint <- factor(clonal_prev_base$timepoint, levels = c("IDX", "MRD"))

# ==========================================
# 2. define every tree edges
# ==========================================
tree_edges <- data.frame(
  source = c("0", "1", "1", "1", "1", "1", "1"),
  target = c("1", "2", "3", "4", "5", "6", "7")
)

# ==========================================
# 3. transform clonal prevalence data
# ==========================================
clonal_prev_final <- clonal_prev_base %>%
  select(timepoint, clone_id, clonal_prev = obs_prop)

# ==========================================
# 4. plot fishplot
# ==========================================
ts_plot <- timescape(
  clonal_prev = clonal_prev_final, 
  tree_edges = tree_edges, 
  height = 600, 
  width = 900
)

ts_plot

In [ ]:
library(htmlwidgets)
library(webshot2)

out_pdf <- paste0(outdir, sample, "_node_stage_fishplot.pdf")

out_html <- paste0(outdir, sample, "_node_stage_fishplot.html")

saveWidget(ts_plot, file = out_html, selfcontained = TRUE)

webshot(out_html, file = out_pdf, delay = 3, vwidth = 1200, vheight = 700)  

#### 3. Fisher test

In [ ]:
library(tidyverse)

df_node_info_qc <- read.csv(paste0(outdir, sample, "_cell_stage.csv"))

df_count <- df_node_info_qc %>%
  group_by(node, Stage) %>%
  summarise(count = n(), .groups = "drop") %>%
  mutate(node = as.character(node))

total_idx <- sum(df_count$count[df_count$Stage == "IDX"])
total_mrd <- sum(df_count$count[df_count$Stage == "MRD"])


final_results <- df_count %>%
  pivot_wider(
    names_from = Stage, 
    values_from = count, 
    values_fill = list(count = 0) 
  ) %>%
  rename(count_IDX = IDX, count_MRD = MRD) %>%
  
  rowwise() %>%
  mutate(
    prop_IDX = count_IDX / total_idx,
    prop_MRD = count_MRD / total_mrd,
    delta_prop = prop_MRD - prop_IDX,
    
    fisher_pvalue = fisher.test(matrix(
      c(count_IDX, total_idx - count_IDX,
        count_MRD, total_mrd - count_MRD),
      nrow = 2
    ))$p.value
  ) %>%
  ungroup() %>%

  mutate(FDR = p.adjust(fisher_pvalue, method = "BH")) %>%
  
  mutate(Significance = case_when(
    FDR < 0.001 ~ "***",
    FDR < 0.01  ~ "**",
    FDR < 0.05  ~ "*",
    TRUE        ~ "ns" 
  )) %>%
  
  select(
    node, 
    count_IDX, count_MRD, 
    prop_IDX, prop_MRD, 
    delta_prop, 
    fisher_pvalue, FDR, Significance
  ) %>%
  arrange(FDR)

print(head(final_results))

output_file <- paste0(outdir, sample, "_clonal_evolution_fisher_fdr.csv")
write.csv(final_results, output_file, row.names = FALSE)